# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [ ]:
# Uncomment in a fresh Kaggle notebook environment.
%pip install -qU \
  torch==2.10.0 \
  torchvision \
  torchaudio \
  transformers==5.2.0 \
  unsloth==2026.3.10 \
  unsloth_zoo==2026.3.4 \
  trl==0.24.0 \
  peft==0.18.1 \
  accelerate==1.12.0 \
  bitsandbytes==0.49.2 \
  xformers==0.0.35 \
  tokenizers==0.22.2 \
  sentencepiece==0.2.1 \
  torchao==0.16.0 \
  datasets pandas lxml cairosvg

In [ ]:
# from google.colab import drive
# import os
# drive.mount('/content/drive')

# import shutil
# if not os.path.exists("./kaggle"):
#     shutil.copytree(
#         "/content/drive/MyDrive/SVG/kaggle",
#         "./kaggle")

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET
from tqdm import tqdm
import tiktoken

import numpy as np
import pandas as pd
import torch

from datasets import concatenate_datasets, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).
CONFIG = {
    "model_name": "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit",  # Verify exact ID from the linked Unsloth notebook.
    "max_seq_length": 2048,
    "lora_r": 16,  # 16
    "lora_alpha": 16,
    "learning_rate": 2e-4,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 24,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 100,
    "max_train_samples_per_source": None,
    "eval_size": 0.02,
    "output_dir": "/kaggle/working/qwen2b_svg_lora",
}

CONFIG

In [ ]:
ET.register_namespace("", "http://www.w3.org/2000/svg")
enc = tiktoken.get_encoding("cl100k_base")

# Setting
MAX_TOKENS         = 1536
MAX_PATHS          = 35
MAX_ELEMENTS       = 75
MAX_RAW_CHARS      = 5000
MAX_PATH_COMMANDS  = 45
MAX_TOTAL_COMMANDS = 80
MAX_ARC_PER_PATH   = 2
MAX_TOTAL_ARC      = 4
MIN_PROMPT_WORDS   = 4
TARGET_W, TARGET_H = 200, 200

FORBIDDEN_TAGS = ["<image","<text","<filter","<mask","<pattern",
                  "<foreignObject","<script","<style","<animate"]

def count_tokens(text):
    return len(enc.encode(text))

def get_root(svg):
    try:
        return ET.fromstring(svg)
    except:
        return None

def local_tag(tag):
    return tag.split("}")[-1] if "}" in tag else tag

def is_valid_svg(svg):
    if not isinstance(svg, str) or not svg.strip().startswith("<svg"):
        return False
    return get_root(svg) is not None

def is_exact_200(svg):
    m = re.search(r'viewBox="([\d\.\- ]+)"', svg)
    if not m:
        return False
    try:
        x, y, w, h = map(float, m.group(1).split())
        return x == 0 and y == 0 and w == TARGET_W and h == TARGET_H
    except:
        return False

def path_complexity_ok(svg):
    root = get_root(svg)
    if root is None:
        return False
    total_cmd = total_arc = 0
    for elem in root.iter():
        if local_tag(elem.tag) != "path":
            continue
        d = elem.get("d", "")
        cmd = len(re.findall(r"[MmLlHhVvCcSsQqTtAaZz]", d))
        arc = len(re.findall(r"[Aa]", d))
        if cmd > MAX_PATH_COMMANDS or arc > MAX_ARC_PER_PATH:
            return False
        total_cmd += cmd
        total_arc += arc
    return total_cmd <= MAX_TOTAL_COMMANDS and total_arc <= MAX_TOTAL_ARC

def remove_noise(svg):
    svg = re.sub(r"<defs.*?</defs>", "", svg, flags=re.S|re.I)
    svg = re.sub(r"<linearGradient.*?</linearGradient>", "", svg, flags=re.S|re.I)
    svg = re.sub(r"<radialGradient.*?</radialGradient>", "", svg, flags=re.S|re.I)
    svg = re.sub(r'\s(id|class|data-[^=]*|version|xml:space|enable-background)="[^"]*"', "", svg, flags=re.I)
    svg = re.sub(r'\s(inkscape:|sodipodi:)[^=]*="[^"]*"', "", svg, flags=re.I)
    svg = re.sub(r'\s+xmlns:xlink="[^"]*"', "", svg, flags=re.I)
    return svg

def compress_numbers(svg):
    def repl(m):
        s = f"{float(m.group()):.2f}".rstrip("0").rstrip(".")
        return s
    return re.sub(r"-?\d+\.\d+", repl, svg)

def canonicalize_svg_200(svg):
    root = get_root(svg)
    if root is None:
        return None
    inner = "".join(ET.tostring(c, encoding="unicode") for c in root)
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" '
        'width="200" height="200" viewBox="0 0 200 200">'
        f'{inner.strip()}</svg>'
    )

def prompt_ok(prompt):
    if not isinstance(prompt, str):
        return False
    p = prompt.strip()
    return (len(p) >= 8 and len(p.split()) >= MIN_PROMPT_WORDS and
            not any(p.startswith(x) for x in
                    ["Generate an SVG","Create an SVG","Draw an SVG","Generate SVG","Create SVG"]))

def clean_sample(prompt, svg):
    if not prompt_ok(prompt):
        return None
    if not isinstance(svg, str):
        return None
    svg = svg.strip()

    if len(svg) > MAX_RAW_CHARS:    return None
    if not is_valid_svg(svg):   return None
    if not is_exact_200(svg):   return None
    if any(t in svg.lower() for t in FORBIDDEN_TAGS):   return None
    if len(re.findall(r"<path\b", svg, re.I)) > MAX_PATHS:  return None
    if sum(1 for _ in get_root(svg).iter()) > MAX_ELEMENTS: return None
    if not path_complexity_ok(svg): return None

    svg = remove_noise(svg)
    if re.search(r'url\(#.+?\)', svg, re.I):    return None

    svg = canonicalize_svg_200(svg)
    if svg is None: return None

    svg = compress_numbers(svg)
    if not is_valid_svg(svg):   return None
    if count_tokens(svg) > MAX_TOKENS:  return None

    return prompt.strip(), svg.strip()

# Run
df = pd.read_csv("/kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv")
clean_data = []

for _, row in tqdm(df.iterrows(), total=len(df)):
    res = clean_sample(row["prompt"], row["svg"])
    if res:
        clean_data.append(res)

print("clean size:", len(clean_data))
clean_df = pd.DataFrame(clean_data, columns=["prompt", "svg"])
clean_df.insert(0, "id", range(len(clean_df)))
clean_df.to_csv("/kaggle/working/train_clean.csv", index=False)

In [ ]:
from datasets import Dataset

df = pd.read_csv("/kaggle/working/train_clean.csv")
df = df[df["svg"].str.startswith("<svg", na=False)].reset_index(drop=True)
df = df.sample(n=13000, random_state=SEED).reset_index(drop=True)
print(f"{len(df)}")

train_raw = Dataset.from_pandas(df[["prompt", "svg"]])
train_raw = train_raw.shuffle(seed=SEED)

splits = train_raw.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds = splits["train"]
eval_ds = splits["test"]

print(f"Train: {len(train_ds)}, Eval: {len(eval_ds)}")

In [ ]:
import matplotlib.pyplot as plt

df["svg_len"] = df["svg"].str.len()
print(df["svg_len"].describe())
print(f"95th percentile: {df['svg_len'].quantile(0.95):.0f} chars")

df["svg_len"].hist(bins=50)
plt.xlabel("SVG length (chars)")
plt.ylabel("Count")
plt.title("SVG Length Distribution")
plt.show()

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element. "
    "Keep the SVG concise and under 4500 characters where possible. "
    "Always include width=\"256\" height=\"256\" viewBox=\"0 0 200 200\" in the <svg> tag."
)

_SVG_HEADER_256 = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 200 200">'

def format_sft_text(example):
    svg = re.sub(r'<svg[^>]*>', _SVG_HEADER_256, example['svg'], count=1)
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{svg}<|im_end|>"
    )
    return {"text": text}

train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

In [ ]:
from transformers import TrainerCallback
from IPython.display import clear_output

_train_log, _eval_log = [], []

class LossPlotCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            _train_log.append((state.global_step, logs["loss"]))

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_loss" in metrics:
            _eval_log.append((state.global_step, metrics["eval_loss"]))
            clear_output(wait=True)
            plt.figure(figsize=(8, 3))
            if _train_log: plt.plot(*zip(*_train_log), label="train")
            if _eval_log:  plt.plot(*zip(*_eval_log), "o-", label="eval")
            best = min(_eval_log, key=lambda x: x[1])
            plt.axvline(best[0], color="g", linestyle="--",
                        label=f"best step={best[0]} loss={best[1]:.4f}")
            plt.legend(); plt.grid(True, alpha=0.4); plt.tight_layout();
            plt.savefig("/kaggle/working/train_loss.png")
            plt.close()

In [ ]:
# from transformers import TrainingArguments
# from trl import SFTTrainer
from unsloth import UnslothTrainer as SFTTrainer
from unsloth import UnslothTrainingArguments as TrainingArguments

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    seed=SEED,
    load_best_model_at_end=True,  # add
    metric_for_best_model="eval_loss",  # add
    greater_is_better=False,  # add
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    #packing=True,
    args=training_args,
    callbacks=[LossPlotCallback()],  # show
)

train_result = trainer.train()
train_result

In [ ]:
from IPython.display import Image
Image("/kaggle/working/train_loss.png")

In [ ]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

In [ ]:
# load model

# from unsloth import FastLanguageModel
# import shutil
# from peft import PeftModel

# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=CONFIG["model_name"],
#     max_seq_length=CONFIG["max_seq_length"],
#     dtype=None,
#     load_in_4bit=True,
# )

# if os.path.exists("/kaggle/working/qwen2b_svg_lora"):
#     shutil.rmtree("/kaggle/working/qwen2b_svg_lora")
# if not os.path.exists("/kaggle/working/qwen2b_svg_lora"):
#     shutil.copytree(
#         "/kaggle/input/datasets/yutaigu/version5/version5/kaggle/working/qwen2b_svg_lora",
#         "/kaggle/working/qwen2b_svg_lora")

# model = PeftModel.from_pretrained(model, "./kaggle/working/qwen2b_svg_lora2")
# FastLanguageModel.for_inference(model)

In [ ]:
from IPython.display import SVG, display, HTML
import cairosvg
from IPython.display import Image

FastLanguageModel.for_inference(model)

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", flags=re.IGNORECASE)

def extract_svg(text):
    if not text:
        return ""
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith("svg")
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

def fix_svg_attrs(svg):
    return re.sub(
        r'<svg[^>]*>',
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 200 200">',
        svg,
        count=1
    )

def fix_svg(svg):
    if not svg:
        return ""
    if not svg.rstrip().endswith("</svg>"):
        svg = svg[:svg.rfind(">")+1] + "</svg>"
    svg = re.sub(r'<svg[^>]*>',
                 '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 200 200">',
                 svg, count=1)
    return svg

def generate_svg(prompt, max_new_tokens=512):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.05,
        )
    input_len = inputs["input_ids"].shape[1]
    decoded = tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True)
    svg = fix_svg(extract_svg(decoded))
    return svg


test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt, max_new_tokens=2048)
print(pred_svg)
print("Valid SVG:", is_valid_svg(pred_svg))
display(SVG(data=pred_svg))

In [ ]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
TEST_PROMPTS_PATH = "/kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

if os.path.exists(SUBMISSION_PATH):
    os.remove(SUBMISSION_PATH)

In [ ]:
# shutil.copy(
#     "/kaggle/input/datasets/yutaigu/version4/submission.csv",
#     "/kaggle/working/submission.csv"
# )

In [ ]:
test_df = pd.read_csv(TEST_PROMPTS_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

MAX_RETRIES = 5

def _build_chat(prompt):
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt} /no_think<|im_end|>\n"
        "<|im_start|>assistant\n"
        "<think>\n\n</think>\n"
    )

def generate_svg_batch(prompts, max_new_tokens=2048, batch_size=4):
    """Returns (svgs, raw_decoded) — two lists of the same length."""
    orig_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"

    all_svgs, all_raws = [], []
    for i in range(0, len(prompts), batch_size):
        batch     = [_build_chat(p) for p in prompts[i : i + batch_size]]
        inputs    = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
        attn_mask = inputs["attention_mask"]
        pos_ids   = (attn_mask.cumsum(dim=1) - 1).clamp(min=0)
        input_len = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            output_ids = model.generate(
                input_ids          = inputs["input_ids"],

                attention_mask     = attn_mask,
                position_ids       = pos_ids,
                max_new_tokens     = max_new_tokens,
                do_sample          = True,
                temperature        = 0.4,
                top_p              = 0.9,
                repetition_penalty = 1.05,
                pad_token_id       = tokenizer.eos_token_id,
            )

        for j, out in enumerate(output_ids):
            decoded = tokenizer.decode(out[input_len:], skip_special_tokens=True)
            all_raws.append(decoded)
            all_svgs.append(fix_svg(extract_svg(decoded)))

    tokenizer.padding_side = orig_padding_side
    return all_svgs, all_raws

def submission_test(temp_test_df, batch_size=50):
    if os.path.exists(SUBMISSION_PATH):
        existing_df = pd.read_csv(SUBMISSION_PATH)
        rows        = existing_df.to_dict("records")
        done_ids    = set(r["id"] for r in rows)
        decode_list = [""] * len(rows)
        print(f"Resume from {len(rows)} entries.")
    else:
        done_ids    = set()
        rows        = []
        decode_list = []

    remaining     = temp_test_df[~temp_test_df["id"].isin(done_ids)].reset_index(drop=True)
    result_svgs   = {}  # id -> svg
    result_raws   = {}  # id -> raw

    pending = list(zip(remaining["id"], remaining["prompt"]))

    for attempt in range(MAX_RETRIES):
        if not pending:
            break

        print(f"Attempt {attempt+1}: {len(pending)} prompts")
        failed = []

        for i in tqdm(range(0, len(pending), batch_size), desc=f"  attempt {attempt+1}"):
            batch      = pending[i : i + batch_size]
            ids        = [x[0] for x in batch]
            prompts    = [x[1] for x in batch]

            print(i)

            svgs, raws = generate_svg_batch(prompts, max_new_tokens=2048, batch_size=batch_size)

            for rid, svg, raw in zip(ids, svgs, raws):
                result_raws[rid] = raw
                if is_valid_svg(svg):
                    result_svgs[rid] = svg
                else:
                    failed.append((rid, prompts[ids.index(rid)]))

            partial_rows = list(rows)
            for _, row in remaining.iterrows():
                rid = row["id"]
                if rid in result_svgs:
                    partial_rows.append({"id": rid, "svg": fix_svg_attrs(result_svgs[rid])})
            pd.DataFrame(partial_rows).to_csv(SUBMISSION_PATH, index=False)

        print(f" -> {len(pending)-len(failed)} saved, {len(failed)} failed")
        pending = failed

    invalid_count = 0
    for _, row in remaining.iterrows():
        rid = row["id"]
        svg = result_svgs.get(rid)
        if svg is None:
            invalid_count += 1
            svg = fallback_svg(rid)
        else:
            svg = fix_svg_attrs(svg)
        rows.append({"id": rid, "svg": svg})
        decode_list.append(result_raws.get(rid, ""))

    sub_df = pd.DataFrame(rows)
    sub_df.to_csv(SUBMISSION_PATH, index=False)

    print(f"Saved : {SUBMISSION_PATH}")
    print(f"Rows  : {len(sub_df)}")
    print(f"Invalid/fallback (after {MAX_RETRIES} retries): {invalid_count}")
    return sub_df, decode_list

sub_batch, sub_decode = submission_test(test_df, batch_size=75)


## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.